# V-Energy Grid 360 — Camada Gold (DE-03)

## 1. Visão Geral e Arquitetura da Camada Gold
A camada **Gold** representa o topo da arquitetura Medalhão. Sua responsabilidade é consolidar, agregar e enriquecer os dados limpos vindos da camada **Silver**, transformando tabelas relacionais em **visões analíticas de negócio**.

### Objetivos do Negócio Atendidos:
1. **Diagnóstico de Receita e Tarifa:** Responder à diretoria sobre a variação do volume faturado (-4%) versus o reajuste tarifário (+10,5%), identificando o impacto do consumo, bandeiras e Geração Distribuída (GD).
2. **Qualidade do Serviço (DEC/FEC):** Calcular os indicadores regulatórios da ANEEL por alimentador e município para controle operacional.
3. **Fila de Inspeção e Risco:** Identificar potenciais perdas não técnicas (fraudes/gatos) cruzando dados de consumo, inspeções anteriores e chamados do call center.
4. **Visão 360° da Unidade Consumidora:** Fonte única de verdade para alimentar o backend do Portal Web e o Agente de IA.

---

### Histórias de Usuário Atendidas neste Notebook:
* **Modelagem de Métricas:** *"Como engenheiro de dados, quero modelar a camada Gold a partir das perguntas de negócio das lideranças..."*
* **Indicadores ANEEL:** *"Como engenheiro de dados, quero calcular DEC e FEC por alimentador conforme a definição da ANEEL..."*
* **Testes de Qualidade:** *"Como engenheiro de dados, quero testes de qualidade automatizados no pipeline..."*

###  Leitura de Controle e Rastreabilidade (Ajuste A1)

**Objetivo:** Capturar os metadados de execução da camada Silver antes do processamento.  
**Impacto:** Permite associar cada carga da Gold ao `run_id` e carimbo de data/hora (`data_processamento`) exato do lote de origem, garantindo auditoria completa entre camadas.

In [0]:
# SEÇÃO 0
from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA_SILVER_DADOS = "silver_publicada"  # schema dos dados de negócio
SCHEMA_SILVER_CONTROLE = "silver"         # schema das tabelas de controle
SCHEMA_GOLD = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")


def assert_columns_exist(df, columns, name):
    missing = sorted(set(columns) - set(df.columns))

    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")



#  AJUSTE A1 — VALIDAÇÃO CRUZADA DE RIGOR MÁXIMO (CODEX REFACTORED)

# se o Databricks Job passar o parâmetro "prod", ele assume "prod". Se executar o notebook manualmente sem definir o parâmetro, ele assume "dev" por omissão
try:
    dbutils.widgets.text("AMBIENTE", "dev", "Ambiente de Execução (dev/prod)")
    AMBIENTE = dbutils.widgets.get("AMBIENTE").strip().lower()
except NameError:
    AMBIENTE = "dev"

PERMITIR_RUN_SINTETICO = AMBIENTE in {"dev", "test"}

TB_PUBLICACAO = f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_publicacao"
TB_SILVER = f"{CATALOG}.{SCHEMA_SILVER_CONTROLE}.controle_silver"

tabelas_controle_existentes = (
    spark.catalog.tableExists(TB_PUBLICACAO)
    and spark.catalog.tableExists(TB_SILVER)
)

if not tabelas_controle_existentes:
    if not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            f" Falha de governança: são necessárias as tabelas "
            f"{TB_PUBLICACAO} e {TB_SILVER}. Gold interrompida."
        )

    agora_utc = datetime.now(timezone.utc)
    run_id_silver = (
        f"DEV_SYNTHETIC_{agora_utc.strftime('%Y%m%d_%H%M%S')}_"
        f"{uuid4().hex[:8]}"
    )
    lote_bronze_silver = "DEV_LOTE"
    data_processamento_silver = agora_utc
    silver_lineage_status = "SINTETICO_SEM_CONTROLE_SILVER"

    print("\n [AVISO DE GOVERNANÇA] Tabelas físicas de controle não encontradas.")
    print(f"   • Ambiente: {AMBIENTE}")
    print(f"   • Run ID sintético: {run_id_silver}")
    print(f"   • Status de linhagem: {silver_lineage_status}")
    print("   • Este ID não comprova a versão da Silver consumida.\n")

else:
    df_publicacao = spark.table(TB_PUBLICACAO)
    df_controle_silver = spark.table(TB_SILVER)

    # A publicação precisa identificar explicitamente run e lote ativos.
    assert_columns_exist(
        df_publicacao,
        ["run_ativo", "lote_ativo"],
        TB_PUBLICACAO
    )

    col_data_publicacao = next(
        (
            c for c in ["publicado_em", "data_processamento", "concluido_em"]
            if c in df_publicacao.columns
        ),
        None
    )

    if col_data_publicacao is None:
        raise ValueError(
            f" {TB_PUBLICACAO}: não encontrei uma coluna de data de publicação "
            "entre publicado_em, data_processamento e concluido_em."
        )

    registro_publicacao = (
        df_publicacao
        .filter(
            F.col("run_ativo").isNotNull()
            & F.col("lote_ativo").isNotNull()
            & F.col(col_data_publicacao).isNotNull()
        )
        .withColumn(
            "_data_publicacao_ts",
            F.to_timestamp(F.col(col_data_publicacao))
        )
        .filter(F.col("_data_publicacao_ts").isNotNull())
        .orderBy(
            F.col("_data_publicacao_ts").desc(),
            F.col("run_ativo").cast("string").desc(),
            F.col("lote_ativo").cast("string").desc()
        )
        .limit(1)
        .collect()
    )

    if not registro_publicacao:
        raise ValueError(
            f" {TB_PUBLICACAO}: não há publicação válida com run_ativo, "
            "lote_ativo e data."
        )

    pub = registro_publicacao[0]
    run_id_silver = pub["run_ativo"]
    lote_bronze_silver = pub["lote_ativo"]
    data_processamento_silver = pub["_data_publicacao_ts"]

    # Resolver aliases do schema de controle histórico e validar sua existência.
    colunas_cs = set(df_controle_silver.columns)

    col_run_cs = next(
        (c for c in ["_run_id", "run_id", "run_ativo"] if c in colunas_cs),
        None
    )
    col_lote_cs = next(
        (c for c in ["_lote_bronze", "lote_bronze", "lote_ativo"]
         if c in colunas_cs),
        None
    )
    col_status_cs = next(
        (c for c in ["status", "situacao"] if c in colunas_cs),
        None
    )

    ausentes = [
        nome for nome, valor in [
            ("coluna de run_id", col_run_cs),
            ("coluna de lote", col_lote_cs),
            ("coluna de status", col_status_cs)
        ]
        if valor is None
    ]

    if ausentes:
        raise ValueError(
            f" {TB_SILVER}: schema inválido; faltam {', '.join(ausentes)}."
        )

    # Compare as chaves como texto para evitar incompatibilidade de tipos.
    df_status_run_lote = (
        df_controle_silver
        .filter(
            (F.col(col_run_cs).cast("string") == F.lit(str(run_id_silver)))
            & (F.col(col_lote_cs).cast("string") == F.lit(str(lote_bronze_silver)))
        )
        .withColumn(
            "_status_norm",
            F.upper(F.trim(F.col(col_status_cs).cast("string")))
        )
    )

    resumo_status = (
        df_status_run_lote
        .agg(
            F.count(F.lit(1)).alias("qtd_registros"),
            F.sum(
                F.when(F.col("_status_norm").isNull(), 1).otherwise(0)
            ).alias("qtd_status_nulos"),
            F.collect_set("_status_norm").alias("status_encontrados")
        )
        .first()
    )

    qtd_registros = resumo_status["qtd_registros"] or 0
    qtd_status_nulos = resumo_status["qtd_status_nulos"] or 0
    status_encontrados = resumo_status["status_encontrados"] or []

    status_concluido = (
        qtd_registros > 0
        and qtd_status_nulos == 0
        and all(s == "CONCLUIDO" for s in status_encontrados)
    )

    if not status_concluido and not PERMITIR_RUN_SINTETICO:
        raise RuntimeError(
            f" Falha de execução: run={run_id_silver}, lote={lote_bronze_silver}, "
            f"status encontrados={status_encontrados}, registros={qtd_registros}, "
            f"status nulos={qtd_status_nulos}. Esperado somente CONCLUIDO."
        )

    if status_concluido:
        silver_lineage_status = "CONTROLE_FISICO"
        print("\n [CONTROLE SILVER] Validação cruzada concluída com sucesso:")
    else:
        silver_lineage_status = "CONTROLE_FISICO_NAO_CONCLUIDO"
        print("\n [AVISO DE GOVERNANÇA] Controle físico encontrado, mas não aprovado:")

    print(f"   • Publicação: {TB_PUBLICACAO}")
    print(f"   • Histórico: {TB_SILVER}")
    print(f"   • Run ID: {run_id_silver}")
    print(f"   • Lote: {lote_bronze_silver}")
    print(f"   • Processada em: {data_processamento_silver}")
    print(f"   • Status encontrados: {status_encontrados}")
    print(f"   • Linhagem: {silver_lineage_status}\n")


 
#  FUNÇÕES AUXILIARES E ASSERTS DE QUALIDADE


def assert_no_nulls(df, columns, name):
    """Falha se houver nulos nas colunas informadas."""
    condition = None

    for column in columns:
        part = F.col(column).isNull()
        condition = part if condition is None else (condition | part)

    if condition is not None and df.filter(condition).limit(1).count() > 0:
        raise ValueError(
            f"{name}: há valores nulos nas colunas obrigatórias {columns}."
        )


def assert_unique(df, columns, name):
    """Falha se a chave informada não for única."""
    duplicates = (
        df.groupBy(*columns)
          .count()
          .filter(F.col("count") > 1)
          .limit(1)
          .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"{name}: a chave {columns} não é única. "
            "Revise a granularidade da fonte antes de continuar."
        )


def get_distinct_uc_with_gd(df_gd):
    """
    Retorna uma linha por UC com GD.

    A regra considera que qualquer registro em geracao_distribuida
    significa que a UC possui GD. Se a fonte tiver status ou vigência,
    substitua essa regra pela condição de negócio apropriada.
    """
    assert_columns_exist(df_gd, ["id_uc"], "geracao_distribuida")

    return (
        df_gd
        .filter(F.col("id_uc").isNotNull())
        .select("id_uc")
        .dropDuplicates(["id_uc"])
        .withColumn("possui_gd", F.lit("Sim"))
    )


print(f"Silver Dados:    {CATALOG}.{SCHEMA_SILVER_DADOS}")
print(f"Silver Controle: {TB_PUBLICACAO} / {TB_SILVER}")
print(f"Gold:            {CATALOG}.{SCHEMA_GOLD}")

### Governança e operação do Job Gold

**Parâmetro de ambiente**

O código não deve ser alterado manualmente entre desenvolvimento e produção. A célula de configuração inicial foi atualizada para criar e ler o widget dinamicamente:

```python
try:
    dbutils.widgets.text("AMBIENTE", "dev", "Ambiente de Execução (dev/prod)")
    AMBIENTE = dbutils.widgets.get("AMBIENTE").strip().lower()
except NameError:
    AMBIENTE = "dev"

In [0]:
# Verifica a estrutura de colunas da tabela de controle encontrada
spark.table("workspace.silver.controle_publicacao").printSchema()

## 2. Diagnóstico de Receita e Variação Tarifária
### Objetivo:
Atender ao pedido do Diretor Comercial explicando a queda no volume faturado e o impacto na receita por município, classe de consumo, subgrupo e presença de geração solar (GD).

### Regras de Negócio e Agregações:
* Cruzamento das tabelas `silver.faturas`, `silver.leituras`, `silver.unidades_consumidoras` e `silver.geracao_distribuida`.
* Métricas geradas: Consumo total , Receita total , Ticket médio e Tarifa Média Efetiva.

In [0]:

# SEÇÃO 1: Construção da Gold Diagnóstico de Receita

df_faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
df_leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
df_ucs = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")
df_gd = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida")

# Validar colunas esperadas nas fontes
assert_columns_exist(df_faturas, ["id_uc", "mes_referencia", "valor_total"], "faturas")
assert_columns_exist(df_leituras, ["id_uc", "mes_referencia", "consumo_kwh"], "leituras")
assert_columns_exist(df_ucs, ["id_uc", "municipio", "classe", "subgrupo_tarifario"], "unidades_consumidoras")

# Base de UCs (uma linha de cadastro por UC, removendo UCs sintéticas de teste)
df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .select("id_uc", "municipio", "classe", "subgrupo_tarifario")
)

assert_unique(df_ucs_base, ["id_uc"], "unidades_consumidoras")

# Agregar faturas por UC e mês
df_faturas_uc_mes = (
    df_faturas
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("valor_total").alias("receita_uc_mes"),
        F.count(F.lit(1)).alias("qtd_faturas_uc_mes")
    )
)

assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "faturas agregadas por UC e competência")

# Agregar leituras por UC e mês
df_leituras_uc_mes = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull() 
        & F.col("mes_referencia").isNotNull()
        & ~F.col("id_uc").startswith("TESTE")
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("consumo_kwh").alias("consumo_uc_mes"),
        F.count(F.col("consumo_kwh")).alias("qtd_leituras_validas_uc_mes")
    )
)

assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "leituras agregadas por UC e competência")

# Unir faturas com leituras e cadastro (INNER JOIN entre faturas e leituras)
df_uc_mes = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "inner")
    .join(df_ucs_base, "id_uc", "left")
)

# Tratar bandeira tarifária se existir em faturas
if "bandeira_tarifaria" in df_faturas.columns:
    df_bandeira_uc_mes = (
        df_faturas
        .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
        .groupBy("id_uc", "mes_referencia")
        .agg(
            F.first("bandeira_tarifaria", ignorenulls=True).alias("bandeira_tarifaria"),
            F.countDistinct("bandeira_tarifaria").alias("qtd_bandeiras_uc_mes")
        )
    )

    df_uc_mes = df_uc_mes.join(df_bandeira_uc_mes, ["id_uc", "mes_referencia"], "left")

    if df_bandeira_uc_mes.filter(F.col("qtd_bandeiras_uc_mes") > 1).limit(1).count() > 0:
        raise ValueError(
            "Há mais de uma bandeira tarifária por UC/competência. "
            "Defina a regra de rateio ou a granularidade correta antes de gerar diagnostico_receita."
        )
else:
    df_uc_mes = df_uc_mes.withColumn("bandeira_tarifaria", F.lit(None).cast("string"))

# Adicionar Flag de GD
df_gd_uc = get_distinct_uc_with_gd(df_gd)

df_uc_mes = (
    df_uc_mes
    .join(df_gd_uc, "id_uc", "left")
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd"), F.lit("Não")))
)

# Métricas agregadas no grão de negócio (Competência/Município/Classe/Subgrupo/Bandeira/GD)
group_columns = [
    "mes_referencia",
    "municipio",
    "classe",
    "subgrupo_tarifario",
    "bandeira_tarifaria",
    "possui_gd"
]

gold_diagnostico_receita = (
    df_uc_mes
    .groupBy(*group_columns)
    .agg(
        F.countDistinct("id_uc").alias("qtd_ucs_distintas_faturadas_no_mes"), # Nomenclatura explícita
        F.sum("qtd_faturas_uc_mes").cast("long").alias("soma_faturamentos_mensais"),                  # Registra volume de faturas do grupo
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.avg("receita_uc_mes").alias("ticket_medio_reais_por_uc"),
        F.avg("consumo_uc_mes").alias("consumo_medio_kwh_por_uc")
    )
    .withColumn(
        "tarifa_media_efetiva",
        F.when(
            F.col("consumo_total_kwh").isNotNull() & (F.col("consumo_total_kwh") != 0),
            F.col("receita_total_reais") / F.col("consumo_total_kwh")
        )
    )
    .withColumnRenamed("mes_referencia", "competencia")
    .withColumnRenamed("classe", "classe_consumo")
)

# Checagens antes da gravação
assert_no_nulls(gold_diagnostico_receita, ["competencia"], "diagnostico_receita")

if gold_diagnostico_receita.filter((F.col("receita_total_reais") < 0) | (F.col("consumo_total_kwh") < 0)).limit(1).count() > 0:
    raise ValueError(
        "diagnostico_receita: foram encontrados valores negativos em receita ou consumo. Revise a regra para estornos e ajustes."
    )

(
    gold_diagnostico_receita
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
)

print("Tabela 'diagnostico_receita' gerada e gravada com sucesso!")

###  Clarificação e Renomeação de Métricas Volumétricas (`diagnostico_receita`)

Em atendimento às revisões finais da camada Gold:

1. **Renomeação de Colunas:**
   * Ajustada a nomenclatura da coluna de acúmulo volumétrico de `qtd_ucs_faturadas` / `ucs_somadas_na_gold` para **`qtd_ucs_distintas_faturadas_no_mes`** e adicionada a **`soma_faturamentos_mensais`**.
2. **Eliminação de Ambiguidade:**
   * Como a tabela `diagnostico_receita` é agregada no grão de Competência/Município/Classe, a contagem de UCs é calculada na granularidade mensal. A nova nomenclatura elimina qualquer ambiguidade relacional entre o acúmulo mensal cumulativo e o total absoluto de UCs únicas cadastradas na rede.
3. **Exclusão de Registros Sintéticos:**
   * Aplicado o filtro de segurança na origem (`id_uc NOT LIKE 'TESTE%'`) para evitar a contaminação de faturamentos fictícios nos indicadores agregados de receita.

## 3. Indicadores Operacionais e Regulação ANEEL (DEC / FEC)
### Objetivo:
Calcular a **Duração Equivalente de Interrupção por Unidade Consumidora (DEC)** e a **Frequência Equivalente de Interrupção por Unidade Consumidora (FEC)** agregadas por Alimentador e Município.

### Definição Oficial (PRODIST Módulo 8 / ANEEL):
* **DEC (Duração Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{DEC} = \frac{\text{Soma das Horas de Interrupção}}{\text{Total de UCs do Alimentador}}$$

* **FEC (Frequência Equivalente de Interrupção por Unidade Consumidora):**
  $$\text{FEC} = \frac{\text{Soma das Ocorrências de Interrupção}}{\text{Total de UCs do Alimentador}}$$

In [0]:
# SEÇÃO 2: Construção da Gold DEC e FEC

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "data_abertura", "duracao_min"],
    "ocorrencias_rede"
)

assert_columns_exist(
    df_ucs,
    ["id_uc", "alimentador", "municipio"],
    "unidades_consumidoras"
)

# Uma linha cadastral por UC é necessária para atribuir alimentador e município.
# Se unidades_consumidoras for histórica, selecione o registro vigente
# para cada UC antes desta validação.
df_ucs_rede = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .filter(~F.col("id_uc").startswith("TESTE"))
    .select(
        "id_uc",
        F.col("alimentador").alias("id_alimentador"),
        "municipio"
    )
)

assert_unique(df_ucs_rede, ["id_uc"], "cadastro de UCs para DEC/FEC")

assert_no_nulls(
    df_ucs_rede,
    ["id_alimentador"],
    "cadastro de UCs para DEC/FEC"
)

# Município é mantido como atributo do alimentador, não como dimensão
# que alteraria a soma e o denominador do indicador.
df_municipio_alimentador = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("municipio").alias("qtd_municipios"),
        F.first("municipio", ignorenulls=True).alias("municipio")
    )
)

if (
    df_municipio_alimentador
    .filter(F.col("qtd_municipios") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há alimentadores associados a mais de um município. "
        "Defina uma regra para o município exibido ou remova essa "
        "dimensão da tabela de indicadores."
    )

# Validar campos essenciais e duração.
df_ocorrencias_validas = (
    df_ocorrencias
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & F.col("data_abertura").isNotNull()
    )
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "ocorrencias_rede contém durações negativas."
    )

# Cada evento por UC deve aparecer uma única vez.
assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

df_ocorrencias_prep = (
    df_ocorrencias_validas
    .withColumn(
        "competencia",
        F.date_format(F.to_timestamp("data_abertura"), "yyyy-MM")
    )
    .withColumn(
        "duracao_horas",
        F.col("duracao_min").cast("double") / F.lit(60.0)
    )
)

# Associar alimentador à ocorrência. O cadastro já foi validado como uma linha por UC.
df_ocorrencias_rede = (
    df_ocorrencias_prep
    .join(df_ucs_rede, "id_uc", "inner")
)

# Numerador por alimentador e competência: duração acumulada e número de eventos UC-ocorrência.
df_numeradores = (
    df_ocorrencias_rede
    .groupBy("id_alimentador", "competencia")
    .agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(
            F.struct("id_uc", "id_ocorrencia")
        ).alias("frequencia_total_interrupcoes"),
        F.sum("duracao_horas")
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Denominador cadastral atual por alimentador.
df_denominadores = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("id_uc").alias("total_ucs_alimentador")
    )
)

gold_indicadores_dec_fec = (
    df_numeradores
    .join(df_denominadores, "id_alimentador", "left")
    .join(df_municipio_alimentador.select("id_alimentador", "municipio"),
          "id_alimentador", "left")
    .withColumn(
        "DEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("duracao_total_interrupcoes_horas")
            / F.col("total_ucs_alimentador")
        )
    )
    .withColumn(
        "FEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("frequencia_total_interrupcoes")
            / F.col("total_ucs_alimentador")
        )
    )
    .select(
        "id_alimentador",
        "municipio",
        "competencia",
        "total_ucs_alimentador",
        "ucs_interrompidas",
        "frequencia_total_interrupcoes",
        "duracao_total_interrupcoes_horas",
        "DEC",
        "FEC"
    )
)

assert_no_nulls(
    gold_indicadores_dec_fec,
    ["id_alimentador", "competencia", "total_ucs_alimentador"],
    "indicadores_dec_fec"
)

if (
    gold_indicadores_dec_fec
    .filter(
        (F.col("total_ucs_alimentador") <= 0)
        | (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec: denominador inválido ou métrica negativa."
    )

(
    gold_indicadores_dec_fec
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
    )
)

print("Tabela indicadores_dec_fec gravada.")

###  Ajuste N2 — Exclusão de UCs de Teste no Denominador de Indicadores (DEC / FEC)

**Descrição da Mudança:**  
Aplicado o filtro `~F.col("id_uc").startswith("TESTE")` no DataFrame `df_ucs_rede`. 

**Motivação & Impacto:**  
Anteriormente, o cadastro continha **200 UCs de teste** (prefixo `TESTE`) que inflavam o denominador total do sistema de **45.000 para 45.200 UCs**. 
Ao incluir essas UCs fictícias no cálculo dos indicadores da rede, o denominador ficava subestimado, gerando distorções de até `0,185 h/UC` no **DEC** e `0,038` no **FEC** em determinados alimentadores.

**Validação do Contrato:**
* **Total de UCs ativas na rede:** `45.000` UCs cravadas (removidas as 200 de teste).
* **UCs de teste restantes:** `0`.

## 4. Fila de Inspeção e Risco de Perdas Não Técnicas
### Objetivo:
Consolidar indicadores de risco e anomalia de consumo por Unidade Consumidora para priorização das equipes de campo no combate a fraudes e furtos.

### Regra de Priorização:
* **ALTA - REINCIDENTE:** UCs com histórico prévio de irregularidades confirmadas.
* **ALTA - CONSUMO ZERO:** UCs sem geração solar conectada que apresentaram consumo zerado no histórico.
* **MÉDIA - NUNCA INSPECCIONADO:** UCs que jamais receberam fiscalização de campo.
* **BAIXA:** Demais UCs operando em padrão regular.

In [0]:
# SEÇÃO 3: Construção da Gold Fila de Inspeção

df_inspecoes = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.inspecoes"
)

df_atendimentos = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.atendimentos"
)

df_leituras = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.leituras"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

df_gd = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida"
)

assert_columns_exist(
    df_inspecoes,
    ["id_uc", "id_inspecao", "resultado"],
    "inspecoes"
)

assert_columns_exist(
    df_atendimentos,
    ["id_uc", "protocolo", "nota_satisfacao"],
    "atendimentos"
)

assert_columns_exist(
    df_leituras,
    ["id_uc", "consumo_kwh"],
    "leituras"
)

# Garantindo que as colunas de dados pessoais existem em unidades_consumidoras
assert_columns_exist(
    df_ucs,
    ["id_uc", "cpf_cnpj", "nome_titular"],
    "unidades_consumidoras"
)

# Funções auxiliares para mascaramento de PII (LGPD)
def mascarar_documento(col_name):
    return F.when(
        F.col(col_name).isNotNull() & (F.length(F.trim(F.col(col_name))) > 0),
        F.concat(
            F.substring(F.col(col_name), 1, 3),
            F.lit(".***.***-"),
            F.substring(F.col(col_name), -2, 2)
        )
    ).otherwise(F.lit("***.***.***-**"))

def mascarar_nome(col_name):
    return F.when(
        F.col(col_name).isNotNull() & (F.length(F.trim(F.col(col_name))) > 0),
        F.concat(
            F.split(F.trim(F.col(col_name)), " ").getItem(0),
            F.lit(" *******")
        )
    ).otherwise(F.lit("TITULAR MASCARADO"))

# Uma linha por UC na base da fila com campos de titularidade devidamente mascarados

# Uma linha por UC na base com mascaramento e filtro de UCs de teste/fictícias

df_ucs_base = (
    df_ucs
    .filter(
        F.col("id_uc").isNotNull()
        & ~F.col("id_uc").startswith("TESTE") # Remove UCs de teste da Visão 360 e Fila
        & F.coalesce(F.col("cpf_cnpj") != "00000000000", F.lit(True)) # Protege nulos contra descarte indevido
    )
    .dropDuplicates(["id_uc"])
    .withColumn("cpf_cnpj", mascarar_documento("cpf_cnpj"))
    .withColumn("nome_titular", mascarar_nome("nome_titular"))
)

assert_unique(df_ucs_base, ["id_uc"], "base da fila de inspeção")

# Validar que cada ID de inspeção identifica uma única inspeção.
df_inspecoes_validas = df_inspecoes.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_inspecao").isNotNull()
)

assert_unique(
    df_inspecoes_validas,
    ["id_inspecao"],
    "inspecoes por id_inspecao"
)

df_insp_summary = (
    df_inspecoes_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_inspecao")
         .alias("total_inspecoes_realizadas"),
        F.countDistinct(
            F.when(
                F.col("resultado") == "IRREGULARIDADE CONFIRMADA",
                F.col("id_inspecao")
            )
        ).alias("total_inspecoes_irregulares")
    )
)

# Contar protocolos distintos para evitar duplicidade por linha.
df_atend_validos = df_atendimentos.filter(
    F.col("id_uc").isNotNull()
    & F.col("protocolo").isNotNull()
)

df_atend_summary = (
    df_atend_validos
    .groupBy("id_uc")
    .agg(
        F.countDistinct("protocolo")
         .alias("total_chamados_callcenter"),
        F.avg("nota_satisfacao")
         .alias("media_satisfacao_atendimento")
    )
)

# Média do consumo por UC; nulos são ignorados pelo avg.
df_consumo_medio = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("consumo_kwh").isNotNull()
    )
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_kwh")
         .alias("media_consumo_historico")
    )
)

df_gd_uc = get_distinct_uc_with_gd(df_gd)

gold_fila_inspecao = (
    df_ucs_base
    .join(df_consumo_medio, "id_uc", "left")
    .join(df_insp_summary, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(
        df_gd_uc
        .withColumnRenamed("possui_geracao_solar", "possui_gd"),
        "id_uc",
        "left"
    )
    .withColumn(
        "total_inspecoes_realizadas",
        F.coalesce(F.col("total_inspecoes_realizadas"), F.lit(0))
    )
    .withColumn(
        "total_inspecoes_irregulares",
        F.coalesce(F.col("total_inspecoes_irregulares"), F.lit(0))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "possui_gd",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "prioridade_inspecao",
        F.when(
            F.col("total_inspecoes_irregulares") > 0,
            "ALTA - HISTORICO IRREGULAR"
        )
        .when(
            F.col("media_consumo_historico").isNull(),
            "REVISAR - SEM HISTORICO DE CONSUMO"
        )
        .when(
            (F.col("media_consumo_historico") == 0)
            & (F.col("possui_gd") == "Não"),
            "ALTA - CONSUMO ZERO"
        )
        .when(
            F.col("total_inspecoes_realizadas") == 0,
            "MEDIA - NUNCA INSPECIONADO"
        )
        .otherwise("BAIXA")
    )
)

assert_unique(
    gold_fila_inspecao,
    ["id_uc"],
    "gold fila_inspecao"
)

assert_no_nulls(
    gold_fila_inspecao,
    ["id_uc", "prioridade_inspecao", "cpf_cnpj", "nome_titular"],
    "gold fila_inspecao"
)

(
    gold_fila_inspecao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"
    )
)

print("Tabela fila_inspecao gravada com sucesso e dados devidamente mascarados.")

###  Regra de Negócio: Tratamento do Histórico de Inspeções e Priorização

Para garantir maior precisão no direcionamento da **Fila de Inspeção (`gold_fila_inspecao`)** e evitar **falsos positivos**, a lógica de classificação por histórico irregular foi estritamente delimitada.

####  Diretrizes de Classificação:
1. **Contagem Estrita (`IRREGULARIDADE CONFIRMADA`):** Apenas os registros com o status **exato** de `IRREGULARIDADE CONFIRMADA` são contabilizados como histórico irregular, qualificando a Unidade Consumidora (UC) para a prioridade **`ALTA - HISTORICO IRREGULAR`**.
2. **Exclusão de Falsos Positivos:** 
   - Status ambíguos ou de código genérico (ex: `CONFIRMADA` com 34 ocorrências) **não** acionam a prioridade ALTA.
   - Status de inspeções sem constatação de infração (ex: `NADA CONSTA` com 22 ocorrências) também são desconsiderados na contagem de irregularidades.

---

####  Auditoria de Dados e Validação Cruzada (Silver vs. Gold)

A validação executada via células de teste confirmou que **100% das 808 UCs** com prioridade ALTA na camada Gold correspondem estritamente aos registros validados da Silver:

| Status Exato na Silver (`resultado`) | Qtd. Ocorrências | Impacto na Fila de Inspeção (Gold) | Classificação Resultante |
| :--- | :---: | :--- | :--- |
| **`IRREGULARIDADE CONFIRMADA`** | **808** | Incrementar histórico de irregularidade | **`ALTA - HISTORICO IRREGULAR`** (808 UCs) |
| **`SEM IRREGULARIDADE`** | **480** | Considerado inspeção sem fraude | **`BAIXA`** (517 UCs no total) |
| **`CONFIRMADA`** *(Status 34)* | **34** | Ignorado (evita falso positivo) | Não aciona prioridade ALTA |
| **`NADA CONSTA`** *(Status 22)* | **22** | Ignorado (sem infração) | Não aciona prioridade ALTA |

*Nota: As demais UCs da camada Gold são distribuídas entre `MEDIA - NUNCA INSPECIONADO` (42.652) e `REVISAR - SEM HISTORICO DE CONSUMO` (1.023) conforme regras cadastrais e de consumo.*

###  Adequação de Privacidade e Mascaramento de Dados (LGPD)

Para atender às diretrizes da **D20** e garantir a conformidade com a LGPD nos ativos que alimentam diretamente o portal de clientes e os serviços de IA, os dados pessoais sensíveis da tabela `unidades_consumidoras` foram tratados e mascarados antes da persistência na camada Gold:

1. **Validação do Schema de Origem:** Incluída a asserção de existência das colunas `cpf_cnpj` e `nome_titular` na fonte Silver.
2. **Mascaramento do CPF/CNPJ:** Ocultação parcial dos documentos mantendo visíveis apenas os 3 primeiros e os 2 últimos dígitos (ex: `123.***.***-89`), preservando a utilidade analítica sem expor o dado pessoal.
3. **Mascaramento do Nome do Titular:** Preservação apenas do primeiro nome do cliente, substituindo o sobrenome por caracteres genéricos (ex: `MARIA *******`).
4. **Garantia de Sanidade:** Atualização do teste de validação `assert_no_nulls` para assegurar que nenhum registro da fila de inspeção seja gravado com campos de titularidade nulos ou vazios.

###  Saneamento de Dados Sintéticos e Validação de Titularidade (LGPD)

Em atendimento às revisões finais da camada Gold:

1. **Exclusão de Registros de Teste (UCs Fictícias):**
   * Aplicado o filtro na base cadastral de origem (`df_ucs_base`) para remover registros de testes de desenvolvimento (`id_uc LIKE 'TESTE%'` e documentos zerados `00000000000`).
   * **Efeito Cascata:** Como tanto a `fila_inspecao` quanto a `visao_360_uc` herdam os dados diretamente de `df_ucs_base`, a limpeza do cadastro garantiu automaticamente que ambas as visões contemplem exclusivamente clientes e unidades consumidoras operacionais reais.
2. **Conformidade LGPD Preservada:**
   * Todas as UCs válidas mantêm o mascaramento estrito nos campos `cpf_cnpj` (ex: `123.***.***-89`) e `nome_titular` (ex: `MARIA *******`), assegurando a aderência aos requisitos de privacidade para consumo analítico.

## 5. Visão 360° da Unidade Consumidora (Tabela Fato/Dimensão Central)
### Objetivo:
Criar a tabela unificada que alimentará o **Portal Web** e o **Agente de IA**, permitindo consultar a história completa de qualquer cliente em uma única leitura.

In [0]:
# SEÇÃO 4: Construção da Gold Visão 360

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "duracao_min"],
    "ocorrencias_rede"
)

# Uma linha por ocorrência sofrida por UC.
df_ocorrencias_validas = df_ocorrencias.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_ocorrencia").isNotNull()
)

assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError("ocorrencias_rede contém durações negativas.")

df_interrupcao_agg = (
    df_ocorrencias_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_ocorrencia")
         .alias("total_interrupcoes_sofridas"),
        F.sum(F.col("duracao_min").cast("double") / F.lit(60.0))
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Construção do DataFrame Visão 360 herdando os dados cadastrais mascarados de df_ucs_base
gold_visao_360 = (
    df_ucs_base
    .join(df_gd_uc, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_interrupcao_agg, "id_uc", "left")
    .join(df_consumo_medio, "id_uc", "left")
    .withColumn(
        "possui_gd",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "total_interrupcoes_sofridas",
        F.coalesce(F.col("total_interrupcoes_sofridas"), F.lit(0))
    )
    .withColumn(
        "duracao_total_interrupcoes_horas",
        F.coalesce(
            F.col("duracao_total_interrupcoes_horas"),
            F.lit(0.0)
        )
    )
)

assert_unique(gold_visao_360, ["id_uc"], "gold visao_360_uc")

# Validação estrita de sanidade garantindo que os dados de titularidade mascarados não sejam nulos
assert_no_nulls(
    gold_visao_360, 
    ["id_uc", "cpf_cnpj", "nome_titular"], 
    "gold visao_360_uc"
)

(
    gold_visao_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
)

print("Tabela visao_360_uc gravada com sucesso e dados devidamente mascarados.")

###  Anonimização na Visão 360 do Cliente (`visao_360_uc`)

A tabela `visao_360_uc` consolida as métricas operacionais, atendimento e histórico de consumo diretamente conectada ao cadastro mascarado (`df_ucs_base`).

* **Garantia LGPD:** Os atributos `cpf_cnpj` e `nome_titular` são propagados já mascarados para permitir a leitura direta por camadas de consumo (portais e ferramentas de IA) sem exposição de PII.
* **Validação de Sanidade:** O teste `assert_no_nulls` garante a integridade da chave primária (`id_uc`) e dos campos de titularidade anonimizados.

In [0]:
# TESTE DE VALIDAÇÃO DE MASCARAMENTO (LGPD) - FILA_INSPECAO & VISAO_360_UC

from pyspark.sql import functions as F

print("--- 1. AMOSTRA DE DADOS MASCARADOS: FILA_INSPECAO ---")
df_fila = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao")
display(df_fila.select("id_uc", "cpf_cnpj", "nome_titular", "prioridade_inspecao").limit(5))

print("--- 2. AMOSTRA DE DADOS MASCARADOS: VISAO_360_UC ---")
df_v360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
display(df_v360.select("id_uc", "cpf_cnpj", "nome_titular", "total_interrupcoes_sofridas").limit(5))

# --- 3. VALIDAÇÃO DE SEGURANÇA CONTRA DADOS EM TEXTO CLARO ---
fila_cpfs_expostos = df_fila.filter(~F.col("cpf_cnpj").contains("*")).count()
fila_nomes_expostos = df_fila.filter(~F.col("nome_titular").contains("*")).count()

v360_cpfs_expostos = df_v360.filter(~F.col("cpf_cnpj").contains("*")).count()
v360_nomes_expostos = df_v360.filter(~F.col("nome_titular").contains("*")).count()

print("\n--- RELATÓRIO DE AUDITORIA LGPD ---")
print(f"[FILA_INSPECAO] CPFs em texto claro: {fila_cpfs_expostos} | Nomes em texto claro: {fila_nomes_expostos}")
print(f"[VISAO_360_UC]  CPFs em texto claro: {v360_cpfs_expostos} | Nomes em texto claro: {v360_nomes_expostos}")

if sum([fila_cpfs_expostos, fila_nomes_expostos, v360_cpfs_expostos, v360_nomes_expostos]) == 0:
    print("\n HOMOLOGADO: 100% dos dados cadastrais nas tabelas Gold estão mascarados!")
else:
    print("\n ALERTA: Existem dados pessoais expostos em texto claro em alguma das tabelas.")

## 6. Novas Tabelas/Visões Temporais para GenAI/Dev (Issue #40)

Nesta seção, implementamos as novas tabelas agregadas projetadas para atender às consultas do agente de IA e backend, permitindo análises granulares de consumo e receita por localização e variação temporal por Unidade Consumidora (UC).

---

###  Tabela 1: Consumo e Receita por Bairro e Mês (`consumo_receita_bairro`)

**Objetivo:** Permitir que a IA responda a perguntas executivas sobre desempenho territorial (ex.: *"Qual bairro mais perdeu volume de consumo?"* ou *"Qual o bairro com maior receita no mês?"*) sem a necessidade de varrer toda a série temporal individual das UCs.

* **Grão dos Dados:** Competência (`competencia`) × Município (`municipio`) × Bairro (`bairro`) × Classe de Consumo (`classe_consumo`).
* **Origem das Informações:** Cruzamento da base de faturamento/leituras válidas da Silver com a base cadastral de UCs.
* **Fórmula / Lógica das Métricas:**
  - `consumo_total_kwh`: $\sum(\text{consumo\_kwh})$ faturado na competência/bairro/classe.
  - `receita_total_reais`: $\sum(\text{valor\_total})$ faturado na competência/bairro/classe.
  - `qtd_ucs_distintas_faturadas_no_mes`: $countDistinct(\text{id\_uc})$ ativas/faturadas no agrupamento do mês.
* **Critério de Aceite e Conciliação:** A soma consolidada de `receita_total_reais` agrupada por competência e classe deve corresponder exatamente ao total da tabela `diagnostico_receita` (R$ 591.560.472,66).

#### Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `competencia` | String / Date | Mês e ano de referência da fatura (formato YYYY-MM) |
| `municipio` | String | Nome do município onde a UC está localizada |
| `bairro` | String | Nome do bairro associado à UC |
| `classe_consumo` | String | Classe de consumo da UC (Residencial, Comercial, Industrial, etc.) |
| `consumo_total_kwh` | Double | Soma do consumo em kWh faturado no mês/bairro/classe |
| `receita_total_reais` | Double | Soma da receita em Reais (R$) faturada no mês/bairro/classe |
| `qtd_ucs_distintas_faturadas_no_mes` | Long | Contagem de UCs únicas faturadas naquela combinação |


In [0]:
# Seção 5.1: tabela 1 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: consumo_receita_bairro ===")

# Confirmar a granularidade das fontes pré-agregadas.
assert_unique(
    df_faturas_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_faturas_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# Garantir uma linha cadastral por UC. Não deduplicar silenciosamente:
# se houver duplicatas, revise a regra de vigência do cadastro.
df_ucs_bairro = (
    df_ucs_base
    .select(
        "id_uc",
        "municipio",
        "bairro",
        F.col("classe").alias("classe_consumo")
    )
)

assert_unique(df_ucs_bairro, ["id_uc"], "cadastro de UCs para consumo por bairro")

# Agregar faturas e leituras já agregadas por UC/mês.
# O cadastro usa LEFT JOIN para não eliminar faturas elegíveis sem dimensão cadastral.
df_consumo_receita_bairro_base = (
    df_faturas_uc_mes
    .join(
        df_leituras_uc_mes,
        ["id_uc", "mes_referencia"],
        "inner"
    )
    .join(df_ucs_bairro, "id_uc", "left")
    .withColumn(
        "municipio",
        F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "bairro",
        F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado"))
    )
    .withColumn(
        "classe_consumo",
        F.coalesce(F.col("classe_consumo").cast("string"), F.lit("Não informado"))
    )
)

gold_consumo_receita_bairro = (
    df_consumo_receita_bairro_base
    .groupBy(
        F.col("mes_referencia").alias("competencia"),
        "municipio",
        "bairro",
        "classe_consumo"
    )
    .agg(
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.countDistinct("id_uc")
         .alias("qtd_ucs_distintas_faturadas_no_mes")
    )
)

assert_no_nulls(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

assert_unique(
    gold_consumo_receita_bairro,
    ["competencia", "municipio", "bairro", "classe_consumo"],
    "consumo_receita_bairro"
)

(
    gold_consumo_receita_bairro
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
)

print("Tabela consumo_receita_bairro gravada na Gold.")

display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro")
         .orderBy("competencia", "municipio", "bairro", "classe_consumo")
         .limit(10)
)

#### Registros de Auditoria e Reconciliação — Tabela 1

 * **Metodologia de Validação:** Foi realizado o somatório do campo `receita_total_reais` agrupado na tabela `consumo_receita_bairro` e confrontado contra a tabela de controle `gold_diagnostico_receita`.
 * **Resultado do Teste de Reconciliação:**
   - **Receita `consumo_receita_bairro`:** R$ 591.560.472,66
   - **Receita `gold_diagnostico_receita`:** R$ 591.560.472,66
   - **Diferença / Divergência:** R$ 0,00 (0,00%)
 * **Status:** **APROVADO**. A conciliação financeira cravou em 100%, garantindo integridade total de dados e ausência de duplicações na junção entre faturas, leituras e cadastro de UCs.

###  Tabela 2: Variação de Consumo por UC (`variacao_consumo_uc`)

**Objetivo:** Permitir que a IA responda a perguntas sobre comportamento temporal individualizado (ex.: *"Quais UCs com painel solar tiveram queda de consumo acima de 30%?"*).

- **Grão dos Dados:** Unidade Consumidora (`id_uc`).
- **Regra de Negócio:**
  - **Período Inicial (P1):** Média de consumo mensal de 2022-07 a 2023-06 (12 meses).
  - **Período Final (P2):** Média de consumo mensal de 2025-07 a 2026-06 (12 meses).
  - **Variação Absoluta (`variacao_kwh`):** $\text{Consumo Médio Final} - \text{Consumo Médio Inicial}$
  - **Variação Percentual (`variacao_pct`):** $\frac{\text{Consumo Médio Final} - \text{Consumo Médio Inicial}}{\text{Consumo Médio Inicial}} \times 100$
- **Tratamento de Nulos (Ajuste A2):** A coluna `variacao_pct` fica estritamente **nula (`NULL`)** apenas quando a UC não possui histórico de leituras em **nenhum** mês do período analisado (P1 e P2).

#### Dicionário de Colunas:

| Coluna | Tipo | Descrição |
| :--- | :--- | :--- |
| `id_uc` | String | Identificador único da Unidade Consumidora |
| `municipio` | String | Município da UC |
| `bairro` | String | Bairro da UC |
| `possui_gd` | String | Indica se possui geração distribuída ("Sim" / "Não") |
| `consumo_medio_inicial` | Double | Consumo médio mensal em kWh no P1 (2022-07 a 2023-06) |
| `consumo_medio_final` | Double | Consumo médio mensal em kWh no P2 (2025-07 a 2026-06) |
| `variacao_kwh` | Double | Variação absoluta de consumo em kWh |
| `variacao_pct` | Double | Variação percentual de consumo (%) — fica `NULL` apenas se não houver histórico de leituras no período |

In [0]:
# Seção 5.2: tabela 2 adicionada
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA: variacao_consumo_uc ===")

# 1. Validação de formato de competência YYYY-MM
if (
    df_leituras_uc_mes
    .filter(
        F.col("mes_referencia").isNull()
        | ~F.col("mes_referencia").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$")
    )
    .limit(1)
    .count() > 0
):
    raise ValueError("df_leituras_uc_mes contém competências nulas ou fora do formato YYYY-MM.")

# 2. Validações de integridade e granularidade da fonte de leituras
assert_no_nulls(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

assert_unique(
    df_leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "df_leituras_uc_mes"
)

# 3. P1: julho/2022 a junho/2023 (Média + Contagem de meses com consumo VÁLIDO)
df_p1 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2022-07", "2023-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_inicial"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p1")
    )
)

# 4. P2: julho/2025 a junho/2026 (Média + Contagem de meses com consumo VÁLIDO)
df_p2 = (
    df_leituras_uc_mes
    .filter(F.col("mes_referencia").between("2025-07", "2026-06"))
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_uc_mes").alias("consumo_medio_final"),
        F.countDistinct(
            F.when(F.col("consumo_uc_mes").isNotNull(), F.col("mes_referencia"))
        ).alias("meses_com_leitura_p2")
    )
)

# 5. Base Cadastral de Referência
df_variacao_base = gold_visao_360.select("id_uc", "municipio", "bairro", "possui_gd")
assert_unique(df_variacao_base, ["id_uc"], "gold_visao_360 para variação")

df_variacao_unida = (
    df_variacao_base
    .join(df_p1, "id_uc", "left")
    .join(df_p2, "id_uc", "left")
    .withColumn("municipio", F.coalesce(F.col("municipio").cast("string"), F.lit("Não informado")))
    .withColumn("bairro", F.coalesce(F.col("bairro").cast("string"), F.lit("Não informado")))
    .withColumn("possui_gd", F.coalesce(F.col("possui_gd").cast("string"), F.lit("Não")))
    .withColumn("meses_com_leitura_p1", F.coalesce(F.col("meses_com_leitura_p1"), F.lit(0)))
    .withColumn("meses_com_leitura_p2", F.coalesce(F.col("meses_com_leitura_p2"), F.lit(0)))
)

# Expressão para variação percentual bruta (não arredondada para máxima precisão da flag)
variacao_pct_bruta = (
    (F.col("consumo_medio_final") - F.col("consumo_medio_inicial")) 
    / F.col("consumo_medio_inicial")
) * 100

# 6. Construção das métricas, arredondamentos finais e flag booleana
gold_variacao_consumo_uc = (
    df_variacao_unida
    .withColumn(
        "variacao_kwh",
        F.when(
            F.col("consumo_medio_inicial").isNotNull() & F.col("consumo_medio_final").isNotNull(),
            F.round(F.col("consumo_medio_final") - F.col("consumo_medio_inicial"), 2)
        )
    )
    .withColumn(
        "variacao_pct_bruta",
        F.when(
            (F.col("consumo_medio_inicial") > 0) & F.col("consumo_medio_final").isNotNull(),
            variacao_pct_bruta
        )
    )
    .withColumn(
        "variacao_pct",
        F.round("variacao_pct_bruta", 2)
    )
    .withColumn(
        "queda_consumo_superior_30_pct",
        F.coalesce(F.col("variacao_pct_bruta") < -30, F.lit(False))
    )
    .drop("variacao_pct_bruta")
    .withColumn("consumo_medio_inicial", F.round("consumo_medio_inicial", 2))
    .withColumn("consumo_medio_final", F.round("consumo_medio_final", 2))
)

# 7. Asserções finais de saída
assert_no_nulls(gold_variacao_consumo_uc, ["id_uc", "municipio", "bairro", "possui_gd"], "variacao_consumo_uc")
assert_unique(gold_variacao_consumo_uc, ["id_uc"], "variacao_consumo_uc")

# 8. Persistência no catálogo Delta Lake
(
    gold_variacao_consumo_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
)

print("Tabela variacao_consumo_uc gravada com sucesso no catálogo Gold.")

# 9. Exibe amostra da tabela publicada lida diretamente do catálogo Delta
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc")
         .orderBy("id_uc")
         .limit(10)
)

####  Registros de Auditoria e Casos de Uso (GenAI) — Tabela 2

 * **Cobertura Base:** A tabela abrange exatamente **45.000 Unidades Consumidoras**, cobrindo 100% da base cadastral da distribuidora.
 * **Períodos Auditados:**
   - **P1 (Média Inicial):** Período de 2022-07 a 2023-06 (12 meses).
   - **P2 (Média Final):** Período de 2025-07 a 2026-06 (12 meses).
 * **Evidência do Caso de Uso (Issue #40):**
   - Foram mapeadas **3.120 UCs** que possuem Geração Distribuída (`possui_gd = 'Sim'`) e apresentaram queda de consumo superior a **30%** (`variacao_pct <= -30.0%`).
 * **Tratamento de Exceções:** UCs sem histórico completo em um dos dois janelas temporais tiveram seus indicadores `variacao_kwh` e `variacao_pct` mantidos como `NULL`, prevenindo distorções em modelos analíticos.
 * **Status:** **APROVADO**. A tabela atende plenamente aos requisitos de consulta temporal individualizada para o backend e agentes de IA.

### Tabela 3: Consumo e Receita Mensal por Unidade Consumidora (`gold_consumo_mensal_uc`)

**Objetivo:**  
Armazenar a série histórica temporal granular de consumo (`kWh`) e receita (`R$`) por Unidade Consumidora (UC) e competência (`mês/ano`). Esta tabela serve como camada intermediária para alimentar a **Visão 360°** e permitir que agentes de GenAI respondam a consultas pontuais de histórico de faturamento e consumo mensal por UC sem necessidade de reprocessar faturas em tempo de execução.

**Especificações da Tabela:**
* **Grão:** Unidade Consumidora (`id_uc`) × Competência (`competencia` / `YYYY-MM`).
* **Chave Primária:** (`id_uc`, `competencia`).
* **Regras de Negócio e Enriquecimento:**
  * **Junção de Fontes:** `INNER JOIN` entre faturas e leituras pré-agregadas por UC/mês, garantindo 100% de alinhamento com a `gold_diagnostico_receita`.
  * **Arredondamento:** Valores monetários (`receita_reais`) e volumétricos (`consumo_kwh`) padronizados em duas casas decimais.
  * **Persistência:** Tabela gravada no catálogo Delta Lake (`saveAsTable`) para rápida indexação e consulta SQL.

In [0]:
from pyspark.sql import functions as F

print("=== CONSTRUINDO A TABELA 3: consumo_mensal_uc ===")

# 1. Validações de integridade nas fontes de entrada
assert_no_nulls(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")
assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "df_faturas_uc_mes")

assert_no_nulls(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")
assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "df_leituras_uc_mes")

# 2. Junção das leituras e faturas por UC e Competência
# Inner Join mantém o alinhamento perfeito de faturamento com o diagnóstico financeiro
gold_consumo_mensal_uc = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "inner")
    .select(
        F.col("id_uc"),
        F.col("mes_referencia").alias("competencia"),
        F.round(F.col("consumo_uc_mes"), 2).alias("consumo_kwh"),
        F.round(F.col("receita_uc_mes"), 2).alias("receita_reais")
    )
)

# 3. Asserções de integridade
assert_no_nulls(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")
assert_unique(gold_consumo_mensal_uc, ["id_uc", "competencia"], "consumo_mensal_uc")

# 4. Gravação no Catálogo Delta
tabela_t3_target = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
(
    gold_consumo_mensal_uc
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_t3_target)
)

print(f" Tabela 3 'consumo_mensal_uc' gravada com sucesso em {tabela_t3_target}!")

# 5. Exibe amostra da tabela gravada
display(
    spark.table(tabela_t3_target)
         .orderBy("id_uc", "competencia")
         .limit(10)
)

## 7. Testes de Qualidade Automatizados e Validação do Pipeline
### Objetivo:
Garantir que os dados gerados na Gold atendem às restrições de integridade, contagem e consistência lógica antes de serem exportados para o banco de dados local da aplicação.

In [0]:
# SEÇÃO 6: Suíte de Testes Automatizados de Qualidade

from pyspark.sql import functions as F

tables_gold_pk = {
    "diagnostico_receita": [
        "competencia",
        "municipio",
        "classe_consumo",
        "subgrupo_tarifario",
        "bandeira_tarifaria",
        "possui_gd"
    ],
    "indicadores_dec_fec": [
        "id_alimentador",
        "competencia"
    ],
    "fila_inspecao": ["id_uc"],
    "visao_360_uc": ["id_uc"]
}

# Confere que todas as tabelas existem e têm registros.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_columns_exist(df, pk_cols, nome_completo)

    if df.limit(1).count() == 0:
        raise ValueError(f"{nome_completo} está vazia.")

    print(f"[PASS] {nome_completo}: possui registros.")

# Chaves compostas não podem conter nulos.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_no_nulls(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave sem nulos.")

# A chave declarada deve ser única.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_unique(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave única.")

# Verificar métricas negativas.
df_dec = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
)

if (
    df_dec
    .filter(
        (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
        | (F.col("frequencia_total_interrupcoes") < 0)
        | (F.col("total_ucs_alimentador") <= 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec contém valores negativos "
        "ou denominadores inválidos."
    )

# DEC e FEC não devem ser nulos nos grupos gerados.
if (
    df_dec
    .filter(F.col("DEC").isNull() | F.col("FEC").isNull())
    .limit(1)
    .count() > 0
):
    raise ValueError("indicadores_dec_fec contém DEC ou FEC nulos.")

print("[PASS] DEC/FEC sem valores negativos, nulos ou denominadores inválidos.")

# Verificar métricas de receita e consumo.
df_receita = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
)

if (
    df_receita
    .filter(
        (F.col("receita_total_reais") < 0)
        | (F.col("consumo_total_kwh") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita contém receita ou consumo negativos. "
        "Confirme se estornos e ajustes devem ser permitidos."
    )

# A tarifa efetiva pode ser nula quando consumo total é zero ou nulo.
if (
    df_receita
    .filter(
        (F.col("consumo_total_kwh") > 0)
        & F.col("tarifa_media_efetiva").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita tem tarifa efetiva nula "
        "apesar de consumo positivo."
    )

print("[PASS] Receita e consumo dentro dos limites definidos.")

# Formato estrito YYYY-MM e competência calendário válida.
competencia_invalida = (
    df_receita
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_invalida > 0:
    raise ValueError(
        "diagnostico_receita contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de receita no formato YYYY-MM.")

# Checagem equivalente para DEC/FEC.
competencia_dec_invalida = (
    df_dec
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_dec_invalida > 0:
    raise ValueError(
        "indicadores_dec_fec contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de DEC/FEC no formato YYYY-MM.")

# Garantir que nenhuma UC foi multiplicada nas tabelas de visão/fila.
for tabela in ["fila_inspecao", "visao_360_uc"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

    assert_unique(df, ["id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

print("[PASS] Uma linha por UC na fila e na visão 360.")
print("Todos os testes de qualidade da Gold foram concluídos.")

In [0]:
from pyspark.sql import functions as F

print("=== TESTES DE QUALIDADE — ISSUE #40 ===")

# Nome das tabelas gravadas no catálogo Delta
TABELA_BAIRRO = f"{CATALOG}.{SCHEMA_GOLD}.consumo_receita_bairro"
TABELA_VARIACAO = f"{CATALOG}.{SCHEMA_GOLD}.variacao_consumo_uc"
TABELA_CONSUMO_MENSAL = f"{CATALOG}.{SCHEMA_GOLD}.consumo_mensal_uc"
TABELA_DIAGNOSTICO = f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
TABELA_VISAO_360 = f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc"

# Tolerâncias do case
TOLERANCIA_REAIS = 0.01
TOLERANCIA_KWH = 0.01

# Carregar tabelas diretamente do catálogo Delta
df_bairro_test = spark.table(TABELA_BAIRRO)
df_variacao_test = spark.table(TABELA_VARIACAO)
df_mensal_test = spark.table(TABELA_CONSUMO_MENSAL)
df_diagnostico_test = spark.table(TABELA_DIAGNOSTICO)

try:
    df_v360_test = spark.table(TABELA_VISAO_360)
except Exception:
    df_v360_test = gold_visao_360

# 1. Volumetria cadastral esperada via contagem distinta no catálogo
EXPECTED_UC_COUNT = df_v360_test.select("id_uc").distinct().count()

# 2. Validação de Schemas
cols_esperadas_bairro = ["competencia", "municipio", "bairro", "classe_consumo", "consumo_total_kwh", "receita_total_reais", "qtd_ucs_distintas_faturadas_no_mes"]
cols_esperadas_variacao = ["id_uc", "municipio", "bairro", "possui_gd", "consumo_medio_inicial", "consumo_medio_final", "meses_com_leitura_p1", "meses_com_leitura_p2", "variacao_kwh", "variacao_pct", "queda_consumo_superior_30_pct"]
cols_esperadas_mensal = ["id_uc", "competencia", "consumo_kwh", "receita_reais"]

assert_columns_exist(df_bairro_test, cols_esperadas_bairro, TABELA_BAIRRO)
assert_columns_exist(df_variacao_test, cols_esperadas_variacao, TABELA_VARIACAO)
assert_columns_exist(df_mensal_test, cols_esperadas_mensal, TABELA_CONSUMO_MENSAL)
print("[PASS] Schemas das três tabelas conferem.")

# 3. Nulos e Unicidade
chave_bairro = ["competencia", "municipio", "bairro", "classe_consumo"]
chave_mensal = ["id_uc", "competencia"]

assert_no_nulls(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_no_nulls(df_variacao_test, ["id_uc", "municipio", "bairro", "possui_gd"], TABELA_VARIACAO)
assert_no_nulls(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)

assert_unique(df_bairro_test, chave_bairro, TABELA_BAIRRO)
assert_unique(df_variacao_test, ["id_uc"], TABELA_VARIACAO)
assert_unique(df_mensal_test, chave_mensal, TABELA_CONSUMO_MENSAL)
print("[PASS] Chaves e dimensões obrigatórias validadas sem nulos e únicas em todas as tabelas.")

# 4. Volumetria Cadastral
total_ucs_variacao = df_variacao_test.select("id_uc").distinct().count()
assert total_ucs_variacao == EXPECTED_UC_COUNT, f"Volumetria divergente: esperado {EXPECTED_UC_COUNT:,}, encontrado {total_ucs_variacao:,}."
print(f"[PASS] Volumetria cadastral cravada em {total_ucs_variacao:,} UCs distintas.")

# 5. Conciliação Físico-Financeira da Tabela 1 (Territorial) e Tabela 3 (Mensal) contra o Diagnóstico
totais_bairro = df_bairro_test.agg(
    F.sum("receita_total_reais").alias("receita_bairro"),
    F.sum("consumo_total_kwh").alias("consumo_bairro")
)

totais_mensal = df_mensal_test.agg(
    F.sum("receita_reais").alias("receita_mensal"),
    F.sum("consumo_kwh").alias("consumo_mensal")
)

totais_diagnostico = df_diagnostico_test.agg(
    F.sum("receita_total_reais").alias("receita_diagnostico"),
    F.sum("consumo_total_kwh").alias("consumo_diagnostico")
)

conciliacao_geral = (
    totais_bairro
    .crossJoin(totais_mensal)
    .crossJoin(totais_diagnostico)
    .withColumn("diff_receita_bairro", F.round(F.col("receita_bairro") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_bairro", F.round(F.col("consumo_bairro") - F.col("consumo_diagnostico"), 4))
    .withColumn("diff_receita_mensal", F.round(F.col("receita_mensal") - F.col("receita_diagnostico"), 4))
    .withColumn("diff_consumo_mensal", F.round(F.col("consumo_mensal") - F.col("consumo_diagnostico"), 4))
)

display(conciliacao_geral)

diff_rec_bairro = conciliacao_geral.select("diff_receita_bairro").first()[0]
diff_kwh_bairro = conciliacao_geral.select("diff_consumo_bairro").first()[0]
diff_rec_mensal = conciliacao_geral.select("diff_receita_mensal").first()[0]
diff_kwh_mensal = conciliacao_geral.select("diff_consumo_mensal").first()[0]

assert abs(diff_rec_bairro) <= TOLERANCIA_REAIS, f"Falha na receita da Tabela 1: diferença de R$ {diff_rec_bairro}"
assert abs(diff_kwh_bairro) <= TOLERANCIA_KWH, f"Falha no consumo da Tabela 1: diferença de {diff_kwh_bairro} kWh"
assert abs(diff_rec_mensal) <= TOLERANCIA_REAIS, f"Falha na receita da Tabela 3: diferença de R$ {diff_rec_mensal}"
assert abs(diff_kwh_mensal) <= TOLERANCIA_KWH, f"Falha no consumo da Tabela 3: diferença de {diff_kwh_mensal} kWh"

print(f"[PASS] Conciliação Físico-Financeira ok para Tabelas 1 e 3 (diferenças zeradas em R$ e kWh).")


#  AJUSTE N1.2 — VALIDAÇÃO DINÂMICA DA ISSUE #40 (SEM VALOR HARDCODED)


queda_solar = (
    df_variacao_test
    .filter(
        (F.col("possui_gd") == "Sim")
        & (F.col("queda_consumo_superior_30_pct") == True)
    )
    .count()
)

print(f"UCs com GD e queda >30%: {queda_solar:,}")

# Validação dinâmica para garantir resiliência em novos lotes em produção
assert queda_solar > 0, " ERRO: Nenhuma UC atendeu ao critério de GD com queda >30%!"

print(f"[PASS] Regra de Negócio validada dinamicamente: {queda_solar:,} UCs com Solar (GD) e queda >30% identificadas.")
print("\n Todos os testes de qualidade das TRÊS TABELAS foram concluídos com 100% de sucesso!")

###  Validação do Pipeline Gold e Reconciliação de Dados (Silver ➔ Gold)

Esta etapa de validação garante a integridade estrutural, a precisão matemática e a rastreabilidade do pipeline da camada **Gold** após a integração com as atualizações da camada **Silver**.

####  Escopo dos Testes Aplicados:
1. **Reconciliação Financeira (Faturas & Consumo):** Compara os totais de receita (`valor_total`) e consumo (`consumo_kwh`) agregados na Gold com a fonte Silver para garantir que os `JOINs` não geraram duplicações de linhas ("efeito leque").
2. **Reconciliação Operacional (DEC/FEC - Ocorrências de Rede):** Recalcula do zero os indicadores de frequência e duração de interrupções por alimentador na Silver e compara diretamente com as métricas consolidadas na Gold.
3. **Invariantes e Métricas Inválidas:** Verifica a ausência de valores nulos (`null`), negativos ou infinitos nas métricas financeiras finais.
4. **Integridade de Cobertura Temporal:** Confirma se todas as competências (mês/ano) tratadas na Silver foram preservadas na Gold.
5. **Consistência de Universo (Visão 360):** Assegura que o universo de Unidades Consumidoras (UCs) na Visão 360 bate com o cadastro ativo.

In [0]:
# DECOMPOSIÇÃO DE FATURAS DA SILVER 

from pyspark.sql import functions as F

faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
ucs = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")

# Normalização de colunas
col_comp_fat = "mes_referencia" if "mes_referencia" in faturas.columns else "competencia"
col_comp_lei = "mes_referencia" if "mes_referencia" in leituras.columns else "competencia"
col_val_fat = "valor_total" if "valor_total" in faturas.columns else "valor_total_reais"

# 1. Chaves de existência de Leituras
chaves_leituras = (
    leituras
    .filter(F.col("id_uc").isNotNull() & F.col(col_comp_lei).isNotNull())
    .select(
        F.col("id_uc"),
        F.col(col_comp_lei).alias("comp_leitura")
    )
    .dropDuplicates(["id_uc", "comp_leitura"])
)

# 2. Chaves de existência de Cadastro de UCs
chaves_ucs = (
    ucs
    .filter(F.col("id_uc").isNotNull())
    .select("id_uc")
    .dropDuplicates(["id_uc"])
)

# 3. Classificação Mutuamente Exclusiva das Faturas
faturas_classificadas = (
    faturas
    .filter(F.col("id_uc").isNotNull() & F.col(col_comp_fat).isNotNull())
    .join(
        chaves_leituras,
        (faturas["id_uc"] == chaves_leituras["id_uc"]) & 
        (F.col(col_comp_fat) == F.col("comp_leitura")),
        "left"
    )
    .withColumn("tem_leitura", F.col("comp_leitura").isNotNull())
    .join(chaves_ucs.withColumn("tem_cadastro", F.lit(True)), "id_uc", "left")
    .withColumn("tem_cadastro", F.coalesce(F.col("tem_cadastro"), F.lit(False)))
    .withColumn(
        "grupo",
        F.when(~F.col("tem_leitura"), "1. sem_leitura_correspondente")
         .when(~F.col("tem_cadastro"), "2. com_leitura_sem_cadastro")
         .otherwise("3. com_leitura_e_cadastro")
    )
)

print("--- DECOMPOSIÇÃO DE FATURAS DA SILVER ---")
display(
    faturas_classificadas
    .groupBy("grupo")
    .agg(
        F.count("*").alias("qtd_linhas_fatura"),
        F.sum(col_val_fat).alias("valor_total_reais")
    )
    .orderBy("grupo")
)

In [0]:
# 1. TESTE DE RECONCILIÇÃO FINANCEIRA E AUDITORIA DE DESCARTE (SILVER vs GOLD)

from pyspark.sql import functions as F

faturas_silver = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")

col_valor_silver = "valor_total_reais" if "valor_total_reais" in faturas_silver.columns else "valor_total"

# 1. Total Bruto de Faturas na Silver (sem filtros)
silver_bruta_agg = faturas_silver.agg(
    F.sum(col_valor_silver).alias("receita_bruta_silver"),
    F.countDistinct("id_uc").alias("total_ucs_cadastradas_silver")
)

# 2. Total Processado e Média Mensal de UCs Faturadas na Gold
gold_agg = gold_receita.agg(
    F.sum("receita_total_reais").alias("receita_gold"),
    F.round(F.avg("qtd_ucs_distintas_faturadas_no_mes"), 0).alias("media_mensal_ucs_gold")
)

# 3. Consolidação da Tabela de Reconciliação
reconciliacao_financeira = silver_bruta_agg.crossJoin(gold_agg).select(
    F.col("receita_bruta_silver"),
    F.col("receita_gold"),
    (F.col("receita_bruta_silver") - F.col("receita_gold")).alias("diferenca_descarte_reais"),
    F.round(((F.col("receita_gold") / F.col("receita_bruta_silver")) * 100), 2).alias("pct_aderencia_gold"),
    F.col("total_ucs_cadastradas_silver"),
    F.col("media_mensal_ucs_gold")
)

print("--- RESULTADO DA RECONCILIÇÃO FINANCEIRA E RASTREABILIDADE DE DESCARTE ---")
display(reconciliacao_financeira)

resumo = reconciliacao_financeira.collect()[0]
print(f"\n[AUDITORIA FINANCEIRA]")
print(f"• Receita Bruta Silver:    R$ {resumo['receita_bruta_silver']:,.2f}")
print(f"• Receita Processada Gold: R$ {resumo['receita_gold']:,.2f}")
print(f"• Receita Descartada:      R$ {resumo['diferenca_descarte_reais']:,.2f}")
print(f"• Taxa de Aproveitamento:  {resumo['pct_aderencia_gold']}%")

#  AJUSTE N1.1 — ASSERÇÃO RÍGIDA DE RECONCILIAÇÃO FÍSICO-FINANCEIRA

# Isola a soma do valor faturado das faturas classificadas como sem leitura
receita_sem_leitura = (
    faturas_classificadas
    .filter(F.col("grupo") == "1. sem_leitura_correspondente")
    .agg(F.sum(col_val_fat))
    .first()[0] or 0.0
)

# Reconciliação Independente: Faturas Silver = Gold + Faturas Sem Leitura
receita_total_explicada = resumo['receita_gold'] + receita_sem_leitura
diferenca_nao_explicada = abs(resumo['receita_bruta_silver'] - receita_total_explicada)

# Trava a execução com erro caso haja qualquer divergência inexplicada maior que R$ 0,01
assert diferenca_nao_explicada <= 0.01, (
    f" FALHA CRÍTICA DE RECONCILIAÇÃO FÍSICO-FINANCEIRA!\n"
    f"• Receita Silver: R$ {resumo['receita_bruta_silver']:,.2f}\n"
    f"• Receita Explicada (Gold + Sem Leitura): R$ {receita_total_explicada:,.2f}\n"
    f"• Divergência Inexplicada: R$ {diferenca_nao_explicada:,.2f}"
)

print(f"\n [PASS] Reconciliação Físico-Financeira cravada, Divergência: R$ {diferenca_nao_explicada:.2f}")

###  Reconciliação Financeira e Rastreabilidade do Descarte de Receita (Silver vs Gold)

**Motivação do Ajuste (Review de Engenharia):**  
Anteriormente, o teste de reconciliação executava o mesmo `INNER JOIN` utilizado na construção da camada Gold. Esse método forçava a correspondência prévia entre `faturas` e `leituras`, resultando em uma divergência zero artificial e ocultando o volume de faturas descartadas pela ausência de integridade relacional.

**Refatoração Aplicada no Teste:**  
O teste de conciliação foi atualizado para comparar o **valor bruto total sem filtros** retido na Silver contra a receita efetivamente integrada na Gold somada às faturas sem leitura (auditoria relacional).

**Resultado da Auditoria Financeira:**
* **Receita Bruta Total (Silver):** R$ 647.055.875,02
* **Receita Processada e Retida (Gold):** R$ 591.560.472,66
* **Variação por Descarte Relacional:** R$ 55.495.402,36 (Diferença de ~R$ 55,5 mi)
* **Taxa de Aproveitamento/Aderência:** 91,42%
* **Cadastro Único de UCs (Silver):** 45.000 Unidades Consumidoras
* **Média de UCs Faturadas por Agrupamento (Gold):** 411 UCs distintas

---

###  Explicação da Métrica de UCs (`qtd_ucs_distintas_faturadas_no_mes`)

A diferença entre o total de **45.000 UCs na Silver** e o valor médio de **411 na Gold** decorre da alteração de **granularidade** entre as camadas:

1. **Camada Silver (`faturas`):** A contagem reflete o total individual e absoluto de Unidades Consumidoras cadastradas na base.
2. **Camada Gold (`diagnostico_receita`):** Esta tabela é agregada por **Município, Classe de Consumo e Competência (Mês)**.
3. **Origem do valor 411:** A coluna `qtd_ucs_distintas_faturadas_no_mes` registra o volume distinto de UCs faturadas em cada combinação de segmento/mês. A média calculada (`F.avg`) indica que cada grupo/categoria na Gold consolida, em média, 411 UCs distintas por competência.

---

###  Origem Técnica e Auditoria da Divergência Financeira

A apuração de decomposição e rastreabilidade na camada Silver comprovou que a variação de **R$ 55.495.402,36 (~8,58%)** é decorrente **exclusivamente do descarte de faturas sem medição física de consumo**:

1. **Faturas sem Leitura Correspondente (Descartadas no INNER JOIN):**
   * **Volume:** 163.048 faturas | **Valor:** R$ 55.495.402,36
   * **Causa:** Como a construção da camada Gold utiliza `INNER JOIN` entre `faturas` e `leituras` (na chave `id_uc` e `mes_referencia`), faturas sem registro de medição/leitura na mesma competência são descartadas.
2. **Faturas sem Cadastro de UC (Inexistentes):**
   * **Volume:** 0 faturas | **Valor:** R$ 0,00
   * **Causa:** Não foram encontradas faturas com leitura que estivessem sem cadastro correspondente na tabela `unidades_consumidoras`.
3. **Faturas Elegíveis Integradas na Gold:**
   * **Volume:** 1.928.920 registros mensais | **Valor:** R$ 591.560.472,66 (100% conciliado com a Gold).

**Conclusão:**  
A reconciliação física e financeira confirmou paridade de **100% (divergência R$ 0,00)** entre as camadas: a Receita Bruta da Silver (`R$ 647.055.875,02`) é explicada exatamente pela soma da Receita Gold (`R$ 591.560.472,66`) com as faturas sem leitura (`R$ 55.495.402,36`). A regra de elegibilidade do `INNER JOIN` foi validada com asserção rígida (`assert`), garantindo a rastreabilidade total dos dados.

In [0]:
# 2. TESTE DE RECONCILIAÇÃO OPERACIONAL (DEC/FEC - QUALIDADE DA REDE)

from pyspark.sql import functions as F

ocorrencias_tbl = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede")
ucs_tbl = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")

alimentador_teste = "ALM-REC-013"
competencia_teste = "2026-01"

print(f"Testando Alimentador: {alimentador_teste} | Competência: {competencia_teste}")

# 1. Filtra as UCs do alimentador desejado (evitando ambiguidades)
ucs_filtro = (
    ucs_tbl
    .select("id_uc", F.col("alimentador").alias("alimentador_uc"))
    .filter(F.col("alimentador_uc") == alimentador_teste)
    .dropDuplicates(["id_uc"])
)

# 2. Junta com ocorrencias e calcula as metricas
base_teste = (
    ocorrencias_tbl
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & (F.date_format(F.col("data_abertura"), "yyyy-MM") == competencia_teste)
    )
    .join(ucs_filtro, "id_uc", "inner")
)

print("\n1. Recalculado do Zero direto da Silver:")
display(
    base_teste.agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(F.struct("id_uc", "id_ocorrencia")).alias("frequencia_recalculada"),
        (F.sum(F.col("duracao_min").cast("double")) / F.lit(60.0)).alias("duracao_horas_recalculada")
    )
)

print("\n2. Métricas gravadas na tabela Gold de DEC/FEC:")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec")
    .filter(
        (F.col("id_alimentador") == alimentador_teste)
        & (F.col("competencia") == competencia_teste)
    )
)

In [0]:
# 3. TESTES COMPLEMENTARES: NULOS, TEMPORAL E VISÃO 360

from pyspark.sql import functions as F

print("--- RUNNING SANITY CHECKS ---")

# 3.1 Teste de Nulos ou Métricas Inválidas
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
invalidos = gold_receita.filter(
    F.col("receita_total_reais").isNull() |
    F.col("consumo_total_kwh").isNull() |
    (F.col("receita_total_reais") < 0) |
    (F.col("consumo_total_kwh") < 0)
).count()

print(f"[TESTE 1] Métricas nulas ou inválidas na Gold: {invalidos} (Esperado: 0)")

# 3.2 Cobertura Temporal de Competências (Identificando dinamicamente a coluna certa)
silver_faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")

col_silver = "competencia" if "competencia" in silver_faturas.columns else "mes_referencia"
col_gold = "competencia" if "competencia" in gold_receita.columns else "mes_referencia"

comp_silver = set(r[col_silver] for r in silver_faturas.select(col_silver).distinct().collect() if r[col_silver])
comp_gold = set(r[col_gold] for r in gold_receita.select(col_gold).distinct().collect() if r[col_gold])

diff_meses = comp_silver - comp_gold
print(f"[TESTE 2] Competências ausentes na Gold: {diff_meses} (Esperado: set())")

# 3.3 Cobertura de UCs na Visão 360
v360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
total_v360 = v360.select("id_uc").distinct().count()
print(f"[TESTE 3] Total de UCs únicas na Visão 360: {total_v360}")

#  AJUSTE N1.3 — INTERRUPÇÃO EXPLÍCITA EM CASO DE FALHA NO PIPELINE

total_divergencias = invalidos + len(diff_meses)

if total_divergencias > 0:
    raise ValueError(
        f" FALHA NO PIPELINE DE DADOS: Foram encontradas {total_divergencias} divergências nos testes complementares "
        f"({invalidos} métricas inválidas e {len(diff_meses)} meses ausentes). "
        f"O Job foi interrompido para evitar a publicação de dados inconsistentes na Camada Gold."
    )

print("\n TODOS OS TESTES PASSARAM COM SUCESSO! A CAMADA GOLD ESTÁ HOMOLOGADA COM 0 DIVERGÊNCIAS.")